# Aula 13 - Detecção de Anomalias

Quando o ponto estranho é o mais importante.

Na aula anterior, vimos que o DBSCAN marca alguns pontos como ruído (`label = -1`). Mas e se justamente esses pontos forem os mais importantes? Nesta prática, vamos explorar métodos de detecção de anomalias, desde técnicas univariadas simples até algoritmos multivariados.

## 1. Preparação

Execute as células em ordem. O notebook usa o dataset Breast Cancer Wisconsin, onde tumores malignos são tratados como anomalias.

In [1]:
# Pacotes necessários para executar este notebook:
# uv pip install numpy pandas plotly scikit-learn

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

DRACULA = {
    'bg': '#282a36',
    'fg': '#f8f8f2',
    'cyan': '#8be9fd',
    'green': '#50fa7b',
    'pink': '#ff79c6',
    'red': '#ff5555',
    'yellow': '#f1fa8c',
    'purple': '#bd93f9',
    'muted': '#6272a4',
}

def dracula_layout(fig, title=None):
    fig.update_layout(
        template='plotly_dark',
        paper_bgcolor=DRACULA['bg'],
        plot_bgcolor=DRACULA['bg'],
        font=dict(color=DRACULA['fg'], size=12),
        title=dict(text=title, x=0.5, font=dict(color=DRACULA['fg'], size=16)) if title else {},
        xaxis=dict(gridcolor=DRACULA['muted'], zerolinecolor=DRACULA['muted']),
        yaxis=dict(gridcolor=DRACULA['muted'], zerolinecolor=DRACULA['muted']),
        legend=dict(font=dict(color=DRACULA['fg'])),
    )
    return fig

PAL = {k: v for k, v in DRACULA.items()}  # atalho

## 2. Dataset real: Breast Cancer Wisconsin

Vamos usar o dataset clássico de câncer de mama do Wisconsin (UCI / sklearn):

- 569 amostras com 30 características extraídas de imagens de punção aspirativa
- Classes: **benigno** (357) e **maligno** (212)

### Premissa importante

Neste experimento, vamos **simular um cenário real de detecção de anomalias (novelty detection)**:
temos acesso apenas aos dados **normais (benignos)** e queremos detectar o que foge do padrão,
**sem nunca ter visto um único caso maligno durante o treinamento**.

Por isso ajustaremos o scaler, o PCA e todos os modelos **exclusivamente nos dados benignos**.
Os casos malignos são usados **apenas na avaliação**, nunca no treino.

Em muitas aplicações reais (fraude financeira, falha mecânica, intrusion detection),
os dados anômalos são raros ou desconhecidos — e é exatamente aí que a detecção de anomalias
se torna indispensável. Este exercício ilustra esse cenário.

- Malignos (target=0) serão tratados como **anomalias**
- Usaremos PCA para reduzir a 2 dimensões e visualizar
- **Compararemos** a detecção usando apenas 2 PCs vs. todas as 30 features originais


In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA

data = load_breast_cancer()
X = data.data
y = data.target

# 0 = maligno, 1 = benigno
benign_mask = (y == 1)

# Fit scaler e PCA APENAS nos dados benignos
scaler = StandardScaler()
scaler.fit(X[benign_mask])
X_scaled = scaler.transform(X)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
pca.fit(X_scaled[benign_mask])
X_pca = pca.transform(X_scaled)

dados = pd.DataFrame(X_pca, columns=["pc1", "pc2"])
dados["anomalia_real"] = 1 - y  # 1 = maligno = anomalia

n_normal = (y == 1).sum()
n_anomalia = (y == 0).sum()

print(f"Total de pontos: {len(dados)}")
print(f"Benignos (treino): {n_normal}  |  Malignos (teste): {n_anomalia}")
print(f"Variância explicada por 2 PCs (dados benignos): {pca.explained_variance_ratio_.sum():.2%}")
print()
print(dados.describe())

Total de pontos: 569
Benignos (treino): 357  |  Malignos (teste): 212
Variância explicada por 2 PCs (dados benignos): 55.92%

              pc1         pc2  anomalia_real
count  569.000000  569.000000     569.000000
mean     1.459392    4.338652       0.372583
std      3.873390    7.228514       0.483918
min     -5.929299   -8.254187       0.000000
25%     -1.272771   -0.621011       0.000000
50%      0.622413    2.099428       0.000000
75%      3.390902    8.233611       1.000000
max     19.557743   42.410433       1.000000


## 3. Visualização inicial (PCA)

Vamos plotar os pontos no espaço reduzido por PCA. Os casos malignos (anomalias reais) estão destacados — podemos compará-los com o que os algoritmos detectarem.

In [3]:
from symtable import Symbol


dados['classe'] = dados['anomalia_real'].map({0: 'Benigno', 1: 'Maligno (anomalia)'})
fig = px.scatter(dados, x='pc1', y='pc2', color='classe',
                 color_discrete_map={'Benigno': DRACULA['cyan'], 'Maligno (anomalia)': DRACULA['red']},
                 opacity=0.6,
                 symbol='classe',
                 symbol_map={'Benigno': 'circle', 'Maligno (anomalia)': 'x'},
                 title='Breast Cancer Wisconsin: PCA dos dados')

fig.update_traces(
    marker=dict(size=8),
    selector=dict(name='Maligno (anomalia)')
)

fig.update_xaxes(title_text='Componente Principal 1')
fig.update_yaxes(title_text='Componente Principal 2')
fig = dracula_layout(fig)
fig.show()

## 4. Método IQR (Interquartile Range)

O método IQR identifica outliers com base nos quartis de uma variável. Pontos fora de `Q1 - 1.5*IQR` ou `Q3 + 1.5*IQR` são considerados anômalos.

Vamos calcular os limites do IQR **apenas com os dados benignos** (o que é "normal") e depois verificar quais pontos (benignos ou malignos) estão fora desses limites.

Além disso, vamos comparar o IQR aplicado apenas às 2 componentes PCA com o IQR aplicado a todas as 30 features originais.

**Atenção**: IQR em alta dimensão tende a marcar muitos pontos como anomalia, pois quanto mais features, maior a chance de pelo menos uma estar fora dos limites.

In [4]:
def limites_iqr(serie):
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

# --- IQR com limites APENAS dos benignos ---
lim_inf_p1, lim_sup_p1 = limites_iqr(dados.loc[benign_mask, "pc1"])
lim_inf_p2, lim_sup_p2 = limites_iqr(dados.loc[benign_mask, "pc2"])
dados["anomalia_iqr_pca"] = (dados["pc1"] < lim_inf_p1) | (dados["pc1"] > lim_sup_p1) \
                           | (dados["pc2"] < lim_inf_p2) | (dados["pc2"] > lim_sup_p2)
n_iqr_pca = dados["anomalia_iqr_pca"].sum()

# IQR com limites dos benignos em TODAS as 30 features
outliers_full = np.zeros(len(dados), dtype=bool)
for i in range(X_scaled.shape[1]):
    lim_inf, lim_sup = limites_iqr(pd.Series(X_scaled[benign_mask, i]))
    outliers_full |= (X_scaled[:, i] < lim_inf) | (X_scaled[:, i] > lim_sup)
dados["anomalia_iqr_full"] = outliers_full
n_iqr_full = outliers_full.sum()

print(f"Anomalias pelo IQR (apenas PC1/PC2): {n_iqr_pca}")
print(f"Anomalias pelo IQR (30 features):    {n_iqr_full}")
print()

q1_p1 = dados.loc[benign_mask, "pc1"].quantile(0.25)
q3_p1 = dados.loc[benign_mask, "pc1"].quantile(0.75)
iqr_p1 = q3_p1 - q1_p1
print(f"PC1 (benignos): Q1={q1_p1:.2f}, Q3={q3_p1:.2f}, IQR={iqr_p1:.2f}")
print(f"  Limites: [{q1_p1 - 1.5*iqr_p1:.2f}, {q3_p1 + 1.5*iqr_p1:.2f}]")

q1_p2 = dados.loc[benign_mask, "pc2"].quantile(0.25)
q3_p2 = dados.loc[benign_mask, "pc2"].quantile(0.75)
iqr_p2 = q3_p2 - q1_p2
print(f"\nPC2 (benignos): Q1={q1_p2:.2f}, Q3={q3_p2:.2f}, IQR={iqr_p2:.2f}")
print(f"  Limites: [{q1_p2 - 1.5*iqr_p2:.2f}, {q3_p2 + 1.5*iqr_p2:.2f}]")

Anomalias pelo IQR (apenas PC1/PC2): 185
Anomalias pelo IQR (30 features):    302

PC1 (benignos): Q1=-2.06, Q3=1.26, IQR=3.32
  Limites: [-7.03, 6.23]

PC2 (benignos): Q1=-1.60, Q3=1.78, IQR=3.38
  Limites: [-6.66, 6.84]


In [5]:
fig = make_subplots(1, 2, subplot_titles=['PC1', 'PC2'])
for i, var in enumerate(['pc1', 'pc2'], 1):
    fig.add_trace(go.Box(y=dados[var], name=var,
                         marker_color=DRACULA['cyan'], line_color=DRACULA['fg'], fillcolor=DRACULA['purple']), row=1, col=i)
fig = dracula_layout(fig, title='Boxplot: detecção de outliers por IQ')
fig.show()

In [6]:
fig = make_subplots(1, 2, subplot_titles=[f'PCA: {n_iqr_pca} anomalias', f'30 features: {n_iqr_full} anomalias'])
for i, col in enumerate(['anomalia_iqr_pca', 'anomalia_iqr_full'], 1):
    for label, mask, cor, sz in [('Normal', ~dados[col], DRACULA['cyan'], 5),
                                   ('Anomalia', dados[col], DRACULA['red'], 6)]:
        fig.add_trace(go.Scatter(
            x=dados.loc[mask, 'pc1'], y=dados.loc[mask, 'pc2'],
            mode='markers',
            marker=dict(color=cor, size=sz, opacity=0.6, symbol='x' if label == 'Anomalia' else 'circle'),
            name=label, showlegend=(i == 1)), row=1, col=i)
    fig.update_xaxes(title_text='PC1', row=1, col=i)
    fig.update_yaxes(title_text='PC2', row=1, col=i)
fig = dracula_layout(fig, title='IQR — limites calculados dos benignos')
fig.show()

## 5. DBSCAN como detector de anomalias

O DBSCAN marca pontos em regiões de baixa densidade como ruído (`label = -1`). Esses pontos são candidatos a anomalias.

Vamos aplicar o DBSCAN duas vezes: (a) apenas nas 2 componentes PCA e (b) em todas as 30 features originais padronizadas.

In [7]:
# --- DBSCAN nas componentes PCA ---
modelo_dbscan = DBSCAN(eps=0.8, min_samples=10)
labels_pca = modelo_dbscan.fit_predict(dados[["pc1", "pc2"]].values)
dados["anomalia_dbscan_pca"] = labels_pca == -1
n_dbscan_pca = dados["anomalia_dbscan_pca"].sum()

# --- DBSCAN em todas as 30 features ---
labels_full = modelo_dbscan.fit_predict(X_scaled)
dados["anomalia_dbscan_full"] = labels_full == -1
n_dbscan_full = dados["anomalia_dbscan_full"].sum()

print(f"DBSCAN anomalias (PC1/PC2 apenas): {n_dbscan_pca}")
print(f"DBSCAN anomalias (30 features):    {n_dbscan_full}")
print()

fig = make_subplots(1, 2, subplot_titles=[f'PCA: {n_dbscan_pca} ruídos', f'30 features: {n_dbscan_full} ruídos'])
for i, col in enumerate(['anomalia_dbscan_pca', 'anomalia_dbscan_full'], 1):
    for label, mask, cor, sz in [('Normal', ~dados[col], DRACULA['cyan'], 5),
                                   ('Ruído', dados[col], DRACULA['red'], 6)]:
        fig.add_trace(go.Scatter(
            x=dados.loc[mask, 'pc1'], y=dados.loc[mask, 'pc2'],
            mode='markers',
            marker=dict(color=cor, size=sz, opacity=0.6, symbol='x' if label == 'Ruído' else 'circle'),
            name=label, showlegend=(i == 1)), row=1, col=i)
    fig.update_xaxes(title_text='PC1', row=1, col=i)
    fig.update_yaxes(title_text='PC2', row=1, col=i)
fig = dracula_layout(fig, title='DBSCAN')
fig.show()

DBSCAN anomalias (PC1/PC2 apenas): 306
DBSCAN anomalias (30 features):    569



## 6. Isolation Forest

O Isolation Forest isola pontos com cortes aleatórios. Pontos anômalos são mais fáceis de isolar — precisam de menos cortes para serem separados.

Vamos treinar o modelo **apenas nos dados benignos** (aprendendo o que é "normal") e depois testar em todos os pontos.

O parâmetro `contamination` indica a proporção esperada de anomalias nos dados.

In [8]:
modelo_if = IsolationForest(
    contamination=0.05,
    random_state=RANDOM_STATE
)

# Fit APENAS nos benignos, depois predizer em todos
# --- Isolation Forest nas componentes PCA ---
modelo_if.fit(dados.loc[benign_mask, ["pc1", "pc2"]].values)
pred_if_pca = modelo_if.predict(dados[["pc1", "pc2"]].values)
dados["anomalia_if_pca"] = pred_if_pca == -1
n_if_pca = dados["anomalia_if_pca"].sum()

# --- Isolation Forest em todas as 30 features ---
modelo_if.fit(X_scaled[benign_mask])
pred_if_full = modelo_if.predict(X_scaled)
dados["anomalia_if_full"] = pred_if_full == -1
n_if_full = dados["anomalia_if_full"].sum()

print(f"Isolation Forest anomalias (PC1/PC2): {n_if_pca}")
print(f"Isolation Forest anomalias (30 feats): {n_if_full}")
print()

fig = make_subplots(1, 2, subplot_titles=[f'PCA: {n_if_pca} anomalias', f'30 features: {n_if_full} anomalias'])
for i, col in enumerate(['anomalia_if_pca', 'anomalia_if_full'], 1):
    for label, mask, cor, sz in [('Normal', ~dados[col], DRACULA['cyan'], 5),
                                   ('Anomalia', dados[col], DRACULA['red'], 6)]:
        fig.add_trace(go.Scatter(
            x=dados.loc[mask, 'pc1'], y=dados.loc[mask, 'pc2'],
            mode='markers',
            marker=dict(color=cor, size=sz, opacity=0.6, symbol='x' if label == 'Anomalia' else 'circle'),
            name=label, showlegend=(i == 1)), row=1, col=i)
    fig.update_xaxes(title_text='PC1', row=1, col=i)
    fig.update_yaxes(title_text='PC2', row=1, col=i)
fig = dracula_layout(fig, title='Isolation Forest — treinado só em benignos')
fig.show()

Isolation Forest anomalias (PC1/PC2): 193
Isolation Forest anomalias (30 feats): 193



## 7. Local Outlier Factor (LOF)

O LOF compara a densidade local de cada ponto com a densidade de seus vizinhos. Um ponto é suspeito se está em uma região muito menos densa que seus vizinhos.

Diferente do DBSCAN, o LOF não cria clusters — ele foca apenas na detecção de anomalias.

Usaremos o modo `novelty=True` para treinar **apenas nos benignos** e depois classificar todos os pontos.

In [9]:
# LOF modo novelty: fit APENAS nos benignos, predizer em todos
# --- LOF nas componentes PCA ---
modelo_lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05, novelty=True)
modelo_lof.fit(dados.loc[benign_mask, ["pc1", "pc2"]].values)
pred_lof_pca = modelo_lof.predict(dados[["pc1", "pc2"]].values)
dados["anomalia_lof_pca"] = pred_lof_pca == -1
n_lof_pca = dados["anomalia_lof_pca"].sum()

# --- LOF em todas as 30 features ---
modelo_lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05, novelty=True)
modelo_lof.fit(X_scaled[benign_mask])
pred_lof_full = modelo_lof.predict(X_scaled)
dados["anomalia_lof_full"] = pred_lof_full == -1
n_lof_full = dados["anomalia_lof_full"].sum()

print(f"LOF anomalias (PC1/PC2): {n_lof_pca}")
print(f"LOF anomalias (30 feats): {n_lof_full}")
print()

fig = make_subplots(1, 2, subplot_titles=[f'PCA: {n_lof_pca} anomalias', f'30 features: {n_lof_full} anomalias'])
for i, col in enumerate(['anomalia_lof_pca', 'anomalia_lof_full'], 1):
    for label, mask, cor, sz in [('Normal', ~dados[col], DRACULA['cyan'], 3),
                                   ('Anomalia', dados[col], DRACULA['red'], 5)]:
        fig.add_trace(go.Scatter(
            x=dados.loc[mask, 'pc1'], y=dados.loc[mask, 'pc2'],
            mode='markers',
            marker=dict(color=cor, size=sz, opacity=0.6, symbol='x' if label == 'Anomalia' else 'circle'),
            name=label, showlegend=(i == 1)), row=1, col=i)
    fig.update_xaxes(title_text='PC1', row=1, col=i)
    fig.update_yaxes(title_text='PC2', row=1, col=i)
fig = dracula_layout(fig, title='LOF — modo novelty, treinado só em benignos')
fig.show()

LOF anomalias (PC1/PC2): 175
LOF anomalias (30 feats): 185



## 8. One-Class SVM

O **One-Class SVM** (Schölkopf et al., 1999) é um método clássico de *novelty detection*
que aprende uma fronteira de decisão ao redor dos dados normais. Ele mapeia os dados
para um espaço de características via **kernel RBF** e encontra a hiperesfera que melhor
separa os pontos normais da origem. Pontos fora dessa fronteira são considerados anômalos.

Vamos treinar o modelo **apenas nos dados benignos** e testar em todos os pontos.
O parâmetro  controla a proporção esperada de anomalias (análogo a ).


In [10]:
# --- One-Class SVM ---
svm_pca = OneClassSVM(nu=0.05, kernel="rbf", gamma="auto")
svm_pca.fit(dados.loc[benign_mask, ["pc1", "pc2"]].values)
pred_svm_pca = svm_pca.predict(dados[["pc1", "pc2"]].values)
dados["anomalia_svm_pca"] = pred_svm_pca == -1
n_svm_pca = dados["anomalia_svm_pca"].sum()

svm_full = OneClassSVM(nu=0.05, kernel="rbf", gamma="auto")
svm_full.fit(X_scaled[benign_mask])
pred_svm_full = svm_full.predict(X_scaled)
dados["anomalia_svm_full"] = pred_svm_full == -1
n_svm_full = dados["anomalia_svm_full"].sum()

print(f"SVM anomalias (PC1/PC2): {n_svm_pca}")
print(f"SVM anomalias (30 feats): {n_svm_full}")


SVM anomalias (PC1/PC2): 285
SVM anomalias (30 feats): 226


## 9. Comparação dos métodos: PCA vs. 30 features

Vamos comparar quantas anomalias cada método detectou — separando os resultados obtidos com apenas 2 PCs daqueles com todas as 30 features originais — e quantas das anomalias reais (malignos) foram capturadas.

In [11]:
from sklearn.metrics import cohen_kappa_score, f1_score, precision_score, recall_score

def metrics(real, prevista):
    f1 = f1_score(real, prevista)
    prec = precision_score(real, prevista)
    sens = recall_score(real, prevista)
    kappa = cohen_kappa_score(real, prevista)
    return sens, prec, f1, kappa

linhas = []
for nome, col in [
    ("IQR (PCA)", "anomalia_iqr_pca"),
    ("IQR (30D)", "anomalia_iqr_full"),
    ("DBSCAN (PCA)", "anomalia_dbscan_pca"),
    ("DBSCAN (30D)", "anomalia_dbscan_full"),
    ("IF (PCA)", "anomalia_if_pca"),
    ("IF (30D)", "anomalia_if_full"),
    ("LOF (PCA)", "anomalia_lof_pca"),
    ("LOF (30D)", "anomalia_lof_full"),
    ("SVM (PCA)", "anomalia_svm_pca"),
    ("SVM (30D)", "anomalia_svm_full"),
]:
    total_anom = dados[col].sum()
    sens, prec, f1, kappa = metrics(dados["anomalia_real"], dados[col])
    linhas.append({"método": nome, "detectadas": total_anom,
                   "sensibilidade": f"{sens:.0%}",
                   "precisão": f"{prec:.0%}",
                   "F1": f"{f1:.0%}",
                   "kappa": f"{kappa:.2f}"
                   })

comparacao = pd.DataFrame(linhas)
print("=== Tabela comparativa ===")
print(comparacao.to_string(index=False))
print()

fig = go.Figure()
f1_vals = [float(f[:-1]) / 100 for f in comparacao['F1']]
kap_vals = [float(k) for k in comparacao['kappa']]
metodos = comparacao['método'].tolist()

fig.add_trace(go.Bar(name='F1', y=metodos, x=f1_vals,
                     orientation='h', marker_color=DRACULA['cyan'],
                     text=comparacao['F1'], textposition='outside'))
fig.add_trace(go.Bar(name='Kappa', y=metodos, x=kap_vals,
                     orientation='h', marker_color=DRACULA['purple'],
                     text=comparacao['kappa'], textposition='outside'))
fig.update_layout(barmode='group', height=500)
fig.update_xaxes(range=[0, 1])
fig = dracula_layout(fig, title='F1-score e Kappa por método')
fig.show()


=== Tabela comparativa ===
      método  detectadas sensibilidade precisão  F1 kappa
   IQR (PCA)         185           79%      91% 85%  0.76
   IQR (30D)         302           94%      66% 77%  0.60
DBSCAN (PCA)         306           90%      62% 74%  0.53
DBSCAN (30D)         569          100%      37% 54%  0.00
    IF (PCA)         193           83%      91% 86%  0.79
    IF (30D)         193           83%      91% 86%  0.79
   LOF (PCA)         175           76%      92% 83%  0.75
   LOF (30D)         185           80%      91% 85%  0.77
   SVM (PCA)         285           88%      65% 75%  0.56
   SVM (30D)         226           93%      88% 90%  0.84



## 10. Discussão

Observe os resultados:

- Qual método tem a **melhor sensibilidade** (capturou mais malignos)?
- Qual método tem a **melhor precisão** (menos falsos positivos entre os detectados)?
- O **F1-score** equilibra os dois — quem tem o melhor?
- **DBSCAN (30D)** tem 100% de sensibilidade, mas com que custo? (marcou TUDO como anomalia!)
- Algum método deixou passar anomalias reais (falso negativo)?

### PCA vs. todas as features

Note como os resultados diferem ao usar apenas 2 PCs vs. todas as 30 features:

- **DBSCAN com 30 features marca TODOS os 569 pontos como ruído** — isso é a **maldição da dimensionalidade**: em 30 dimensões, as distâncias entre todos os pontos se tornam muito grandes, então o $ eps=0,8 $ não consegue agrupar nenhum ponto. O recall é 100% inútil — não serve para nada porque precisão é apenas 37% (212/569). F1 = 54%, kappa = 0, ou seja, acertou tudo por acaso, já que classificou tudo como maligno, ele vai acertar 100% dos casos de câncer maligno.
- **DBSCAN (PCA)** tem recall de 90% e precisão de 62%, muito melhor. $ F1 = 74\% $.
- **SVM (30D)** com kernel RBF treinado só em benignos alcança **93% recall e 88% precisão** — F1 de **90%**, o melhor resultado!
- **IQR (30D)** tem 94% recall mas apenas 66% precisão: muitos benignos são falsos positivos.
- A variância retida pelo PCA caiu de 63% para **56%** ao treinar só nos benignos os malignos inflavam artificialmente a variância.

**Morale**: Nunca olhe apenas para sensibilidade (recall). Um modelo que marca tudo como anomalia tem 100% de recall, mas é inútil. O **F1-score** equilibra recall e precisão.


### F1 vs. Kappa de Cohen

O Kappa de Cohen mede a concordância entre predição e realidade descontando o acerto esperado ao acaso.
Enquanto o F1 penaliza mais fortemente os falsos positivos (precisão), o Kappa penaliza modelos que acertam por acaso,
como o **DBSCAN (30D)** que ao marcar tudo como anomalia tem $ F1=54\% $ mas $ \kappa = 0,00 $ (nenhuma
concordância além do acaso). O melhor Kappa é do **SVM (30D)** com 0,84 ($ F1=90\% $) — concordância quase perfeita.
O IQR (PCA) também se sai bem: $ \kappa = 0,69 $.

Consulte a tabela e o gráfico de barras para comparar F1 e Kappa lado a lado. O Kappa é especialmente útil para expor modelos que acertam por acaso (DBSCAN 30D: Kappa = 0,00).

Não existe método perfeito. A escolha depende:

- do custo de falsos positivos vs. falsos negativos
- da natureza dos dados
- do contexto de negócio

### Pergunta para reflexão

Os pontos detectados (malignos) parecem **erro de medição**, **evento raro** ou **caso clinicamente importante**? Justifique com base nos resultados.

In [12]:
def tp_fn_fp_tn(real_col, pred_col):
    r = dados[real_col].values
    p = dados[pred_col].values
    tp = (r == 1) & (p == True)
    fp = (r == 0) & (p == True)
    fn = (r == 1) & (p == False)
    tn = (r == 0) & (p == False)
    out = np.full(len(r), "", dtype=object)
    out[tp] = "maligno verdadeiro"
    out[fp] = "benigno falso"
    out[fn] = "maligno falso"
    out[tn] = "benigno verdadeiro"
    return out, tp, fp, fn

selecao = [
    ("IQR (PCA)", "anomalia_iqr_pca"),
    ("IF (PCA)", "anomalia_if_pca"),
    ("LOF (PCA)", "anomalia_lof_pca"),
    ("DBSCAN (PCA)", "anomalia_dbscan_pca"),
    ("IF (30D)", "anomalia_if_full"),
    ("LOF (30D)", "anomalia_lof_full"),
    ("SVM (30D)", "anomalia_svm_full"),
]

cat_cmap2 = {
    "maligno verdadeiro": DRACULA["red"],
    "benigno falso": DRACULA["yellow"],
    "maligno falso": DRACULA["muted"],
    "benigno verdadeiro": DRACULA["cyan"],
}

titles = []
metricas = []
for nome, col in selecao:
    cats, tp, fp, fn = tp_fn_fp_tn("anomalia_real", col)
    sens = tp.sum() / (tp.sum() + fn.sum()) if (tp.sum() + fn.sum()) > 0 else 0
    prec = tp.sum() / (tp.sum() + fp.sum()) if (tp.sum() + fp.sum()) > 0 else 0
    metricas.append((nome, col, sens, prec))
    titles.append(f"{nome}<br>Sens: {sens:.0%}  Prec: {prec:.0%}")

fig = make_subplots(3, 3, subplot_titles=titles,
                     vertical_spacing=0.12, horizontal_spacing=0.08)
for idx, (nome, col, sens, prec) in enumerate(metricas):
    row, col_idx = idx // 3 + 1, idx % 3 + 1
    cats, _, _, _ = tp_fn_fp_tn("anomalia_real", col)
    for cat in cat_cmap2:
        mask = cats == cat
        if mask.sum() == 0:
            continue
        fig.add_trace(go.Scatter(
            x=dados.loc[mask, "pc1"], y=dados.loc[mask, "pc2"],
            mode="markers",
            marker=dict(color=cat_cmap2[cat], size=3, opacity=0.7),
            name=cat, showlegend=(idx == 0)), row=row, col=col_idx)
    fig.update_xaxes(title_text="PC1", row=row, col=col_idx, title_font_size=9)
    fig.update_yaxes(title_text="PC2", row=row, col=col_idx, title_font_size=9)

fig = dracula_layout(fig, title="Acertos e erros de cada método")
fig.update_layout(height=700)
fig.update_annotations(font_size=9)
fig.show()
